1.  Data Augmentation — create realistic variations of training images.
2.  Regularization — prevent the neural network from memorizing the training data.

Complete CIFAR-10 PROJECT Implementing Data Augmentation & Regularization

Proper Training Pipeline

CIFAR-10
   ↓
Data Augmentation
   ↓
  CNN
   ↓
Dropout
   ↓
Weight Decay
   ↓
Learning Rate Scheduler
   ↓
Early Stopping
   ↓
Best Model

#### Import Libraries

In [178]:
import torch
import torch.nn as nn

from torchvision import datasets, transforms
from torch.utils.data import (
    DataLoader, random_split, Subset
)

#### Configure Device

In [179]:
device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu


#### Define Normalization

In [180]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

#### Training Transform

In [181]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5), # data augmentation
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
    ),

    transforms.ToTensor(),
    transforms.Normalize(
        CIFAR10_MEAN,
        CIFAR10_STD
    )

])

#### Validation Transform

In [182]:
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        CIFAR10_MEAN,
        CIFAR10_STD
    )
])

#### Download CIFAR-10

In [183]:
train_dataset_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

Files already downloaded and verified


NOTE: We need to be careful here.

If we split this dataset directly, the validation subset will also use the training augmentation.

For a proper pipeline, we want:

Training subset → augmentation
Validation subset → deterministic transform

So we'll create the dataset twice.

#### Create Separate Dataset Objects

In [184]:
full_train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

full_val_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=val_transform
)

# Both contain the same underlying CIFAR-10 training samples.

# But they have different transforms.

Files already downloaded and verified
Files already downloaded and verified


#### Train & Validation Split

CIFAR-10 contains:

50,000 training images

Let's use:

45,000 → training
5,000  → validation

In [185]:
train_size = 45_000
val_size = 5_000

train_indices, val_indices = random_split(
    range(len(full_train_dataset)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) # split is reproducible
)

#### Create Subsets

In [186]:
train_dataset = Subset(
    full_train_dataset,
    train_indices.indices
)

val_dataset = Subset(
    full_val_dataset,
    val_indices.indices
)

#### Create DataLoaders

In [187]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

#### Build the CNN Model

In [188]:
# no 43
class CIFAR10CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 8 * 8,
                128
            ),

            nn.ReLU(),

            nn.Dropout(p=0.4), # model regularization

            nn.Linear(
                128,
                10
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

#### Initialize the Model

In [189]:
model = CIFAR10CNN().to(device)

print(model)

CIFAR10CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


#### Loss Function

In [190]:
# CIFAR10 has 10 classes
criterion = nn.CrossEntropyLoss()

#### Optimizer with Weight Decay

In [191]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4 # optimizer regularization
)

#### Learning Rate Scheduler

In [192]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=2
)

# this will monitor validation loss

#### Training Function

In [193]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() # clear previous gradients

        outputs = model(images) # forward pass

        loss = criterion(outputs, labels) # calculate loss

        loss.backward() # calculate gradients

        optimizer.step() # update model weights

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / total

    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

#### the complete cycle
Input
 ↓
Forward
 ↓
Prediction
 ↓
Loss
 ↓
Backward
 ↓
Gradients
 ↓
Optimizer
 ↓
Updated weights

#### Validation Function

In [194]:
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    epoch_loss = running_loss / total

    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

#### Early Stopping Variables

In [195]:
num_epochs = 3

best_val_loss = float('inf')  # any normal validation loss should be smaller than infinity.

patience = 5

epochs_without_improvement = 0

#### Training Loop

In [196]:
for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f} "
        f"LR: {current_lr:.6f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            "09_best_cifar10_model.pth"
        )

        print("Best model saved.")

    else:

        epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):

            print("Early stopping triggered.")

            break

Epoch [1/3] Train Loss: 1.6695 Train Acc: 0.3901 Val Loss: 1.3599 Val Acc: 0.5092 LR: 0.001000
Best model saved.
Epoch [2/3] Train Loss: 1.4254 Train Acc: 0.4861 Val Loss: 1.1822 Val Acc: 0.5772 LR: 0.001000
Best model saved.
Epoch [3/3] Train Loss: 1.3097 Train Acc: 0.5293 Val Loss: 1.0830 Val Acc: 0.6130 LR: 0.001000
Best model saved.


#### Loading the Best Model

In [197]:
model.load_state_dict(
    torch.load(
    "09_best_cifar10_model.pth",
    map_location=device,
    weights_only=True
    )
)

<All keys matched successfully>